# Dumbarton FC — Sponsor Intelligence Tool · Colab runner

Build the ranked sponsor queue from **real local-business data**, right here in Colab.

**Why this works in Colab** (when the recruitment scraper doesn't): the sponsor tool
reads from *official APIs* — Google Places and Companies House — which don't block
datacenter IPs. So the whole pipeline runs in the cloud, no local setup needed.

**What this notebook does**
1. Get the code · 2. Install · 3. API keys · 4. (optional) persist to Drive ·
5. Build the queue · 6. View it · 7. Inspect a prospect · 8. Update status ·
9. Build the club website · 10. (optional) Streamlit working view.

> Runs with **no keys** on built-in offline fixtures, so you can try the whole flow first.

## 1 · Get the code
Downloads the latest version of the tool from GitHub (`th1555/TH_DFC_Sponsors`). Nothing to edit: just run it.

In [ ]:
import os
REPO_URL = "https://github.com/th1555/TH_DFC_Sponsors.git"

if os.path.basename(os.getcwd()) == "sponsor_tool":   # re-run: already inside, get the latest
    !git pull -q
elif os.path.exists("sponsor_tool"):                 # downloaded earlier this session
    !git -C sponsor_tool pull -q
    os.chdir("sponsor_tool")
else:                                                # first run
    !git clone -q "$REPO_URL" sponsor_tool
    os.chdir("sponsor_tool")
print("working dir:", os.getcwd())
print("modules here:", sorted(f for f in os.listdir() if f.endswith((".py", ".yaml"))))

## 2 · Install dependencies

In [ ]:
!pip install -q -r requirements.txt 2>/dev/null || pip install -q streamlit pandas numpy pyyaml requests
print("dependencies ready")

## 3 · API keys (via Colab Secrets)
Click the **🔑 key icon** in the left sidebar and add any of these as secrets
(toggle *Notebook access* on):

| Secret name | Cost | Get it from |
|---|---|---|
| `COMPANIES_HOUSE_API_KEY` | **free** | developer.company-information.service.gov.uk |
| `GOOGLE_PLACES_API_KEY` | needs Cloud billing (free monthly credit covers low volume) | console.cloud.google.com → Places API |
| `ANTHROPIC_API_KEY` | paid | console.anthropic.com |

Skip any of them and that part falls back to offline fixtures.

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ["GOOGLE_PLACES_API_KEY", "COMPANIES_HOUSE_API_KEY", "ANTHROPIC_API_KEY"]:
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass  # not in Colab

has_places_ch = bool(os.environ.get("GOOGLE_PLACES_API_KEY") and
                     os.environ.get("COMPANIES_HOUSE_API_KEY"))
has_llm = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Places + Companies House:", "LIVE" if has_places_ch else "offline fixtures")
print("LLM soft features:       ", "LIVE (Haiku)" if has_llm else "offline stub")

## 4 · (Optional) Persist the database in Google Drive
Colab's disk resets when the runtime recycles. Mount Drive to keep `sponsor.db` — **and your outreach notes** — across sessions.

In [ ]:
USE_DRIVE = False   # set True to persist the store in your Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/dumbarton_sponsor", exist_ok=True)
    os.environ["DUMBARTON_SPONSOR_DB"] = "/content/drive/MyDrive/dumbarton_sponsor/sponsor.db"
    print("store ->", os.environ["DUMBARTON_SPONSOR_DB"])
else:
    print("using an ephemeral sponsor.db (fine for a quick run; resets on recycle)")

## 5 · Build the queue
Discovers businesses near the ground, enriches them via Companies House, derives the soft features, scores, and stores. Uses live APIs where keys are set, fixtures otherwise.

In [ ]:
import importlib
import sponsor_db as db;        importlib.reload(db)
import sponsor_pipeline as P;   importlib.reload(P)

ingest_mode = "api" if has_places_ch else "fixture"
soft_mode   = "llm" if has_llm else "stub"

conn = P.run(mode=soft_mode, source="real", ingest_mode=ingest_mode)
q = db.get_queue(conn)
print(f"built {len(q)} businesses  |  discovery={ingest_mode}  |  soft features={soft_mode}")
print("stored at:", db.DB_PATH)

## 6 · The ranked queue
Work it top-down. Red bar = fit score.

In [ ]:
import pandas as pd
cols = ["name", "sector", "fit_score", "distance_km", "tier", "status", "flags"]
(q[cols].style
   .bar(subset=["fit_score"], vmin=0, vmax=100, color="#c8102e")
   .format({"fit_score": "{:.0f}", "distance_km": "{:.1f}"})
   .hide(axis="index"))

## 7 · Inspect a prospect
Every score breaks down into its reasons, and each AI judgment cites a fact from the business's own website.

In [ ]:
def inspect(name):
    row = q[q.name == name].iloc[0]
    biz, sf, contrib = db.get_detail(conn, row.business_id)
    print(f"{name}  —  fit {row.fit_score:.0f}  |  {row.sector}  |  {row.distance_km:.1f} km")
    print(f"tier:  {row.tier}")
    print(f"flags: {row.flags}")
    print("\nwhy (points out of each weight):")
    for k, v in sorted(contrib.items(), key=lambda x: -x[1]):
        print(f"   {k.replace('_',' '):<20} {v:5.1f}")
    print("\nLLM soft reads:")
    for _, r in sf.iterrows():
        print(f"   {r.feature:<20} value={r.value:.2f} conf={r.confidence:.2f}  \"{r.justification}\"")

# a dropdown if ipywidgets is available, else just the top prospect
try:
    import ipywidgets as w
    w.interact(inspect, name=w.Dropdown(options=list(q.name), description="Prospect"))
except Exception:
    inspect(q.iloc[0]["name"])

## 8 · Work a prospect (writes to the walled human layer)
Setting a status touches **only** `outreach_status` — a re-build never overwrites it. Statuses: `to-approach`, `contacted`, `in-talks`, `won`, `dead`.

In [ ]:
TARGET = q.iloc[0]["name"]     # change to any business name
db.set_status(conn, q[q.name == TARGET].iloc[0].business_id,
              "to-approach", owner="Tom", notes="Warm path via board — call first.")

db.get_queue(conn).query("status != 'new'")[["name", "status", "owner", "notes", "updated_at"]]

## 9 · Build the club website
Writes the same static page design as Sons Scouting: `site/index.html` (the page the club uses: recorded facts and Local fit) and `site/lab.html` (adds the website readings, prototype only). The preview below is the real page; select any business to open its profile.

In [ ]:
import html, os, shutil
!python export_sponsor_site.py

PAGE = "index.html"        # or "lab.html" for the prototype view
page = open(os.path.join("site", PAGE), encoding="utf-8").read()
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:820px;border:1px solid #ddd;border-radius:8px"></iframe>'))

Download the site folder to deploy it the same way as Sons Scouting (drop it into the Scouting repo's `site/sponsors/` folder and push). Keep that repo private: the pages include outreach notes.

In [ ]:
shutil.make_archive("sons_sponsors_site", "zip", "site")
try:
    from google.colab import files
    files.download("sons_sponsors_site.zip")
except ImportError:
    print("site zipped to sons_sponsors_site.zip")

## 10 · (Optional) The full Streamlit app, inside Colab
Launches the rich action-queue UI behind a temporary Cloudflare tunnel and prints a
public URL. The link is throwaway (changes each run) and effectively single-user —
great for a look, not a deployment.

In [ ]:
import subprocess, time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

app = subprocess.Popen(["streamlit", "run", "sponsor_app.py",
                        "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501",
                        "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("opening tunnel — your URL will appear below (ending in trycloudflare.com):\n")
for line in tun.stdout:
    if "trycloudflare.com" in line:
        print("  ", line.strip().split()[-1]); break
# to stop later: app.terminate(); tun.terminate()